# 09 — Per-stream compute, retry and telemetry accounting

Select the **local** `.venv-local` kernel. Aggregate every structured telemetry invocation and response-record attempt without combining distinct model/condition/representation streams. Preparation, training and inference remain separate phases. Pilot work is separate, Colab billing is excluded, and unavailable historical transfer timings are reported as unavailable rather than assigned to inference.

## 1. Local environment and lineage check

Verify the frozen experiment before reading telemetry or response records.

**Recovery / safe rerun:** Rerun section 1 after a local kernel restart.

**Cost:** No paid API requests.



In [1]:
from pathlib import Path
import sys
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'streamlined/common.py').is_file()), None)
if ROOT is None:
    ROOT = next(p/'streamlined_experiment' for p in candidates if (p/'streamlined_experiment/streamlined/common.py').is_file())
sys.path.insert(0, str(ROOT))
from streamlined.common import *
print('Experiment:', ROOT)
print('Interpreter:', sys.executable)

from streamlined.common import verify_experiment
print('Frozen:',verify_experiment(ROOT)['config_hash'])

Experiment: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment
Interpreter: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\.venv-local\Scripts\python.exe
Frozen: 49e13df7571913d2c39067dfd7dcde93353110b2f4c7523bc8a18de2459b4b73


## 2. Refresh response exports and build auditable compute summaries

First refresh all available response exports, including declared partial streams. Then create the invocation inventory, per-row performance audit, exact-stream resource summaries, retry/coverage counts, model-level resource totals, API token/cache usage and separate per-stream CSVs. Retries and resumes are summed only within the exact stream. Compute files are organized under `results/final/compute/` in `invocations`, `rows`, and `stream_summary` subfolders. Every filename is prefixed with its source notebook (`01`--`07`); cross-notebook summaries use `09`. Model totals are resource totals, not elapsed time.

**Recovery / safe rerun:** Rerun sections 1–2 after new or recovered runs. Outputs rebuild locally; no GPU or paid API work occurs.

**Cost:** No paid API requests.



In [2]:
from continuation.consolidate import consolidate
from continuation.compute import summarize
print('Response export:',consolidate(ROOT,include_optional=True))
report=summarize(ROOT)
print(report)
print('Results manifest:',ROOT/'results/final/compute/09__manifest__compute_files.csv')

Response export: {'files_exported': 132, 'complete_files': 125, 'incomplete_files': 7, 'expected_required_candidate_files': 60, 'expected_optional_candidate_files': 25, 'final_folder': 'd:\\reit4841_code\\1-58-bit-llm-CAM-LDS\\streamlined_experiment\\results\\final'}
{'output_folder': 'd:\\reit4841_code\\1-58-bit-llm-CAM-LDS\\streamlined_experiment\\results\\final', 'telemetry_invocations': 199, 'response_records': 25425, 'exact_stream_phases': 114, 'compute_files': 297, 'manifest': 'd:\\reit4841_code\\1-58-bit-llm-CAM-LDS\\streamlined_experiment\\results\\final\\compute\\09__manifest__compute_files.csv'}
Results manifest: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\results\final\compute\09__manifest__compute_files.csv


## 3. Inspect principal report tables

Display the stream accounting and coverage/retry tables from their categorized compute subfolders.

**Recovery / safe rerun:** Rerun section 3 any time after section 2; it is read-only.

**Cost:** No paid API requests.



In [ ]:
import csv

def preview(folder, name, columns, limit=10):
    path=ROOT/'results/final/compute'/folder/name
    with path.open(encoding='utf-8-sig',newline='') as handle:
        rows=list(csv.DictReader(handle))
    print(f'\n{name}: {len(rows)} rows')
    print(' | '.join(columns))
    for row in rows[:limit]:print(' | '.join(str(row.get(column,'')) for column in columns))

preview('stream_summary','09__compute__stream_summary.csv',['model','condition','representation','phase','invocation_count','experimental_resource_gpu_hours','completed','total_retries'])
preview('rows','09__compute__coverage_and_retries.csv',['model','condition','representation','record_count','completed','context_unsupported','retryable_error','total_retries'])
preview('stream_summary','09__compute__model_resource_summary.csv',['model','representation','phase','stream_count','invocation_count','resource_gpu_hours','estimated_energy_wh'])